Data Access

# Silver Layer - Data Cleaning & Transformation

This notebook processes raw CRM data from the Bronze layer and performs data cleaning and transformations.

### Key Steps:
- Handle missing values
- Standardize column formats
- Remove duplicates
- Prepare data for Gold layer analytics



In [ ]:
from pyspark.sql.functions import sum, col, when

# Read data from Bronze layer (raw files stored in Data Lake)

In [ ]:
spark.conf.set(
  "fs.azure.account.key.storageaccount.dfs.core.windows.net",
  "<Using secure access method (key removed for security)>"
)

df_accounts = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://bronze@storageaccount.dfs.core.windows.net/accounts.csv")

# Display schema and sample data to understand structure

In [ ]:
df_accounts = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://bronze@storageaccount.dfs.core.windows.net/accounts.csv")


df_data_dictionary = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://bronze@storageaccount.dfs.core.windows.net/data_dictionary.csv")


df_products = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://bronze@storageaccount.dfs.core.windows.net/products.csv")


df_sales_pipeline = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://bronze@storageaccount.dfs.core.windows.net/sales_pipeline.csv")


df_sales_teams = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://bronze@storageaccount.dfs.core.windows.net/sales_teams.csv")

df_accounts.show()
df_data_dictionary.show()
df_produts.show()
df_sales_pipeline.show()
df_sales_teams.show()

+----------------+-------------+----------------+-------+---------+---------------+----------------+
|         account|       sector|year_established|revenue|employees|office_location|   subsidiary_of|
+----------------+-------------+----------------+-------+---------+---------------+----------------+
|Acme Corporation|    technolgy|            1996|1100.04|     2822|  United States|            NULL|
|      Betasoloin|      medical|            1999| 251.41|      495|  United States|            NULL|
|        Betatech|      medical|            1986| 647.18|     1185|          Kenya|            NULL|
|      Bioholding|      medical|            2012| 587.34|     1356|     Philipines|            NULL|
|         Bioplex|      medical|            1991| 326.82|     1016|  United States|            NULL|
|        Blackzim|       retail|            2009| 497.11|     1588|  United States|            NULL|
|   Bluth Company|    technolgy|            1993|1242.32|     3027|  United States|Acme Cor

# Clean and standardize column values
# Example: trimming spaces, fixing inconsistent naming

In [ ]:
print(df_accounts.columns)
print(df_data_dictionary.columns)
print(df_products.columns)
print(df_sales.columns)
print(df_sales_teams.columns)

['account', 'sector', 'year_established', 'revenue', 'employees', 'office_location', 'subsidiary_of']
['Table', 'Field', 'Description']
['product', 'series', 'sales_price']
['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage', 'engage_date', 'close_date', 'close_value']
['sales_agent', 'manager', 'regional_office']


In [ ]:
df_accounts = df_accounts \
    .withColumnRenamed("account_name", "account") \
    .withColumnRenamed("sector", "industry") \
    .withColumnRenamed("year_established", "established_year") \
    .withColumnRenamed("revenue", "annual_revenue") \
    .withColumnRenamed("employees", "employee_count") \
    .withColumnRenamed("office_location", "headquarters") \
    .withColumnRenamed("subsidiary_of", "parent_company")

In [ ]:
df_data_dictionary = df_data_dictionary \
    .withColumnRenamed("Table", "table") \
    .withColumnRenamed("Field", "field") \
    .withColumnRenamed("Description", "description")

In [ ]:
df_products = df_products.toDF(
    "product", "series", "price"
)

In [ ]:
df_sales_pipeline = df_sales_pipeline \
    .withColumnRenamed("product", "product") \
    .withColumnRenamed("account", "account") \
    .withColumnRenamed("engage_date", "engagement_date")

In [ ]:
df_sales_teams = df_sales_teams \
    .withColumnRenamed("manager", "manager") \
    .withColumnRenamed("regional_office", "region")

In [ ]:
print(df_accounts.columns)
print(df_data_dictionary.columns)
print(df_products.columns)
print(df_sales.columns)
print(df_sales_teams.columns)

['account', 'industry', 'established_year', 'annual_revenue', 'employee_count', 'headquarters', 'parent_company']
['table', 'field', 'description']
['product', 'series', 'price']
['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage', 'engage_date', 'close_date', 'close_value']
['sales_agent', 'manager', 'region']


# Check for missing values in dataset

In [ ]:
null_counts_df_account = df_accounts.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_accounts.columns])
null_counts_df_data_dictionary = df_data_dictionary.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_data_dictionary.columns])
null_counts_df_products = df_products.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_products.columns])
null_counts_df_sales_pipeline = df_sales_pipeline.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_sales_pipeline.columns])
null_counts_df_sales_teams = df_sales_teams.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_sales_teams.columns])


null_counts_df_account.show()
null_counts_df_data_dictionary.show()
null_counts_df_products.show()
null_counts_df_sales_pipeline.show()
null_counts_df_sales_teams.show()

+-------+--------+----------------+--------------+--------------+------------+--------------+
|account|industry|established_year|annual_revenue|employee_count|headquarters|parent_company|
+-------+--------+----------------+--------------+--------------+------------+--------------+
|      0|       0|               0|             0|             0|           0|            70|
+-------+--------+----------------+--------------+--------------+------------+--------------+

+-----+-----+-----------+
|table|field|description|
+-----+-----+-----------+
|    0|    0|          0|
+-----+-----+-----------+

+-------+------+-----+
|product|series|price|
+-------+------+-----+
|      0|     0|    0|
+-------+------+-----+

+--------------+-----------+-------+-------+----------+---------------+----------+-----------+
|opportunity_id|sales_agent|product|account|deal_stage|engagement_date|close_date|close_value|
+--------------+-----------+-------+-------+----------+---------------+----------+----------

# Note: Missing values in close_value and close_date are expected (ongoing deals)

# Handling  NULL values in dataset


In [ ]:
df_accounts =df_accounts.fillna({
  "parent_company": "Independet"
})

df_sales_pipeline = df_sales_pipeline.fillna({
  "account": "Unknown"
})

# Only valid sales records are retained for further analysis

In [ ]:
null_counts_df_account = df_accounts.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_accounts.columns])
null_counts_df_data_dictionary = df_data_dictionary.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_data_dictionary.columns])
null_counts_df_products = df_products.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_products.columns])
null_counts_df_sales_pipeline = df_sales_pipeline.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_sales_pipeline.columns])
null_counts_df_sales_teams = df_sales_teams.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_sales_teams.columns])


null_counts_df_account.show()
null_counts_df_data_dictionary.show()
null_counts_df_products.show()
null_counts_df_sales_pipeline.show()
null_counts_df_sales_teams.show()

+-------+--------+----------------+--------------+--------------+------------+--------------+
|account|industry|established_year|annual_revenue|employee_count|headquarters|parent_company|
+-------+--------+----------------+--------------+--------------+------------+--------------+
|      0|       0|               0|             0|             0|           0|             0|
+-------+--------+----------------+--------------+--------------+------------+--------------+

+-----+-----+-----------+
|table|field|description|
+-----+-----+-----------+
|    0|    0|          0|
+-----+-----+-----------+

+-------+------+-----+
|product|series|price|
+-------+------+-----+
|      0|     0|    0|
+-------+------+-----+

+--------------+-----------+-------+-------+----------+---------------+----------+-----------+
|opportunity_id|sales_agent|product|account|deal_stage|engagement_date|close_date|close_value|
+--------------+-----------+-------+-------+----------+---------------+----------+----------

# Write cleaned data to Silver layer (Parquet format)
# This data will be used for analytics in Gold layer

In [ ]:
df_accounts.write.mode("overwrite") \
    .format("parquet") \
    .save("abfss://silver@storageaccount.dfs.core.windows.net/accounts")

df_data_dictionary.write.mode("overwrite") \
    .format("parquet") \
    .save("abfss://silver@storageaccount.dfs.core.windows.net/data_dictionary")

df_products.write.mode("overwrite") \
    .format("parquet") \
    .save("abfss://silver@storageaccount.dfs.core.windows.net/products")

df_sales_pipeline.write.mode("overwrite") \
    .format("parquet") \
    .save("abfss://silver@storageaccount.dfs.core.windows.net/sales_pipeline")

df_sales_teams.write.mode("overwrite") \
    .format("parquet") \
    .save("abfss://silver@storageaccount.dfs.core.windows.net/sales_teams")

## Summary

- Data cleaned and standardized
- Missing values handled appropriately
- Data prepared for Gold layer transformations